# Pipeline Final de Inferencia Thoracolumbar Explicado - Colab

Este notebook empaqueta la mejor version actual del pipeline para inferencia sobre
nuevas radiografias.

## Pipeline final actual

1. modelo binario localiza la columna
2. modelo multiclase segmenta `T1..T12 + L1..L5`
3. estimador especializado predice la ultima vertebra visible
4. clipping anatomico recorta la mascara final

## Objetivo del notebook

Permitir inferencia reproducible sobre una carpeta de imagenes nuevas y exportar:

- etiquetas presentes antes y despues del clipping
- ultima vertebra visible estimada
- visualizaciones de apoyo
- mascaras exportables

## Nota importante

Esta version asume que la primera vertebra visible del problema util suele comenzar
en `T1`, y concentra la correccion anatomica en la **ultima vertebra visible**,
porque ese fue el principal cuello de botella observado durante el proyecto.

## 0. Preparacion de Colab

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print('No se detecto entorno Colab o Drive ya estaba montado:', exc)

def resolve_project_root() -> Path:
    env_override = os.environ.get('PROJECT_ROOT_OVERRIDE', '').strip()
    candidates = []
    if env_override:
        candidates.append(Path(env_override))
    candidates.extend([
        Path('/content/drive/MyDrive/dataRadiografias'),
        Path('/content/drive/MyDrive/CodexProjects/dataRadiografias'),
        Path.cwd(),
    ])
    for candidate in candidates:
        if (candidate / 'indice_dataset.csv').exists():
            return candidate
    return candidates[0]


PROJECT_ROOT = resolve_project_root()
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'No existe PROJECT_ROOT={PROJECT_ROOT}. Ajusta esta ruta a tu carpeta real en Google Drive.'
    )

os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())

## 1. Configuracion de inferencia

Ajusta solo esta seccion para usar el pipeline sobre tus imagenes nuevas.

In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path.cwd()
INDEX_PATH = ROOT / 'indice_dataset.csv'
DICT_PATH = ROOT / 'diccionario_etiquetas_T1_T12_L1_L5.json'

def resolve_existing(*relative_candidates: str) -> Path:
    search_roots = [ROOT, ROOT / 'reports']
    for base in search_roots:
        for rel in relative_candidates:
            candidate = base / rel
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'No se encontro ninguno de estos archivos: {relative_candidates}')


def resolve_optional(*relative_candidates: str) -> Path | None:
    search_roots = [ROOT, ROOT / 'reports']
    for base in search_roots:
        for rel in relative_candidates:
            candidate = base / rel
            if candidate.exists():
                return candidate
    return None


MANIFEST_PATH = resolve_existing('analysis_outputs/training_manifest_thoracolumbar.csv')
BINARY_GROUP_MAP_PATH = resolve_optional(
    'analysis_outputs/training_runs_binary_thoracolumbar/binary_spine_group_partition_map.csv'
)
BINARY_MODEL_PATH = resolve_existing('models/binary_spine_thoracolumbar_best.pt')
MULTICLASS_MODEL_PATH = resolve_existing('models/thoracolumbar_partial_cascade_explained_best.pt')
LAST_VISIBLE_MODEL_PATH = resolve_existing('models/last_visible_estimator_thoracolumbar_best.pt')

INFERENCE_IMAGE_DIR = ROOT / 'inference_inputs'
OUTPUT_DIR = ROOT / 'analysis_outputs' / 'final_inference_pipeline_thoracolumbar'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INFERENCE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

for path in [INDEX_PATH, DICT_PATH, MANIFEST_PATH, BINARY_MODEL_PATH, MULTICLASS_MODEL_PATH, LAST_VISIBLE_MODEL_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'No existe archivo requerido: {path}')

SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'

IMG_SIZE_BINARY = (512, 256)
IMG_SIZE_MULTICLASS = (640, 320)
IMG_SIZE_LAST = (384, 192)
BINARY_THRESHOLD = 0.50
ROI_PAD_X = 28
ROI_PAD_Y = 44
MIN_FOREGROUND_PIXELS = 24
IGNORE_INDEX = 255

PROFILE_BINS = 24
PRESENCE_THRESHOLD_PIXELS = 40
ASSUMED_FIRST_VISIBLE_IDX = 0  # T1
SAVE_MASKS = True
SAVE_FIGURES = True
MAX_VIS_PREVIEW = 10
USE_DEMO_DATASET_IMAGES_IF_INPUT_EMPTY = True
DEMO_IMAGE_COUNT = 5
MAX_AUX_NORM_SAMPLES = 96

print('DEVICE:', DEVICE)
print('INFERENCE_IMAGE_DIR:', INFERENCE_IMAGE_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('BINARY_GROUP_MAP_PATH:', BINARY_GROUP_MAP_PATH)

## 2. Metadata y clases del problema

Esta seccion solo se usa para:

- reconstruir las clases del problema
- recalcular la normalizacion de features auxiliares del modelo final

In [ ]:
index_df_raw = pd.read_csv(INDEX_PATH)
manifest_df = pd.read_csv(MANIFEST_PATH)
group_map_df = pd.read_csv(BINARY_GROUP_MAP_PATH) if BINARY_GROUP_MAP_PATH is not None else pd.DataFrame()
with open(DICT_PATH, 'r', encoding='utf-8') as f:
    labels_dict = json.load(f)

index_col_map = {
    'grupo': 'split',
    'imagen': 'image',
    'id_paciente': 'patient_id',
    'ruta_radiografia': 'radiograph_path',
    'ruta_mascara_binaria': 'label_binary_path',
    'ruta_mascara_multiclase_id_png': 'multiclass_id_png',
}
index_df = index_df_raw.rename(columns=index_col_map).copy()

final_multiclass_map = {int(k): v for k, v in labels_dict['mascara_multiclase_id_png'].items()}
class_names = [final_multiclass_map[i] for i in range(len(final_multiclass_map))]
num_classes = len(class_names)
canonical_labels = [f'T{i}' for i in range(1, 13)] + [f'L{i}' for i in range(1, 6)]
label_to_class_id = {label: idx for idx, label in enumerate(class_names)}

join_cols = ['split', 'image', 'patient_id', 'radiograph_path']
dataset_subset = index_df[join_cols + ['label_binary_path', 'multiclass_id_png']].copy()
table = manifest_df.merge(dataset_subset, on=join_cols, how='left', suffixes=('', '_idx'))
table['multiclass_mask_path'] = table['mask_path'].fillna(table['multiclass_id_png'])
table['radiograph_path_abs'] = table['radiograph_path'].apply(lambda rel: str((ROOT / rel).resolve()))
table['binary_mask_path_abs'] = table['label_binary_path'].apply(lambda rel: str((ROOT / rel).resolve()))

for col in ['usable_for_thoracolumbar_core', 'usable_for_thoracolumbar_partial', 'needs_annotation_review']:
    if col in table.columns:
        table[col] = table[col].map(
            lambda x: x if isinstance(x, bool) else str(x).strip().lower() == 'true'
        )

if not group_map_df.empty and {'group_id_for_split', 'partition'}.issubset(group_map_df.columns):
    group_partition_map = group_map_df.drop_duplicates().set_index('group_id_for_split')['partition'].to_dict()
    table['partition'] = table['group_id_for_split'].map(group_partition_map)
else:
    table['partition'] = ''

usable_partial_mask = table['usable_for_thoracolumbar_partial'] & ~table['needs_annotation_review']
train_norm_df = table.loc[
    usable_partial_mask & table['partition'].eq('train')
].copy().reset_index(drop=True)

if train_norm_df.empty:
    train_norm_df = table.loc[usable_partial_mask].copy().reset_index(drop=True)
    print('No se encontro split train para aux normalization. Se usara todo el subset partial usable.')

if len(train_norm_df) > MAX_AUX_NORM_SAMPLES:
    train_norm_df = train_norm_df.sample(MAX_AUX_NORM_SAMPLES, random_state=SEED).reset_index(drop=True)

print('Train samples for aux normalization:', len(train_norm_df))

## 3. Utilidades de imagen y arquitecturas del pipeline

In [ ]:
def read_gray(path: str) -> np.ndarray:
    return np.array(Image.open(path).convert('L'))


def resize_image(arr: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    return np.array(Image.fromarray(arr).resize((size[1], size[0]), resample=Image.BILINEAR))


def resize_mask(arr: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    return np.array(Image.fromarray(arr.astype(np.uint8)).resize((size[1], size[0]), resample=Image.NEAREST))


def build_binary_mask(path: str, size: tuple[int, int] | None = None) -> np.ndarray:
    mask = read_gray(path)
    mask = (mask >= 127).astype(np.uint8)
    if size is not None:
        mask = resize_mask(mask, size)
    return mask


def bbox_from_mask(mask: np.ndarray, min_foreground_pixels: int = 24) -> tuple[int, int, int, int] | None:
    ys, xs = np.where(mask > 0)
    if len(xs) < min_foreground_pixels:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def clamp_bbox(bbox: tuple[int, int, int, int], image_shape: tuple[int, int]) -> tuple[int, int, int, int]:
    h, w = image_shape
    x0, y0, x1, y1 = bbox
    x0 = max(0, min(x0, w - 1))
    y0 = max(0, min(y0, h - 1))
    x1 = max(x0 + 1, min(x1, w))
    y1 = max(y0 + 1, min(y1, h))
    return x0, y0, x1, y1


def expand_bbox(bbox: tuple[int, int, int, int], image_shape: tuple[int, int], pad_x: int = 28, pad_y: int = 44) -> tuple[int, int, int, int]:
    x0, y0, x1, y1 = bbox
    return clamp_bbox((x0 - pad_x, y0 - pad_y, x1 + pad_x, y1 + pad_y), image_shape)


def scale_bbox(bbox: tuple[int, int, int, int], src_shape: tuple[int, int], dst_shape: tuple[int, int]) -> tuple[int, int, int, int]:
    src_h, src_w = src_shape
    dst_h, dst_w = dst_shape
    x0, y0, x1, y1 = bbox
    sx = dst_w / src_w
    sy = dst_h / src_h
    scaled = (int(round(x0 * sx)), int(round(y0 * sy)), int(round(x1 * sx)), int(round(y1 * sy)))
    return clamp_bbox(scaled, dst_shape)


def crop_array(arr: np.ndarray, bbox: tuple[int, int, int, int]) -> np.ndarray:
    x0, y0, x1, y1 = bbox
    return arr[y0:y1, x0:x1]


def normalize_image(image_2d: np.ndarray) -> np.ndarray:
    mean = float(image_2d.mean())
    std = float(image_2d.std())
    if std < 1e-6:
        return image_2d - mean
    return (image_2d - mean) / std


def build_coordinate_channels(height: int, width: int) -> np.ndarray:
    y_coords = np.linspace(0.0, 1.0, height, dtype=np.float32)[:, None]
    x_coords = np.linspace(0.0, 1.0, width, dtype=np.float32)[None, :]
    y_map = np.repeat(y_coords, width, axis=1)
    x_map = np.repeat(x_coords, height, axis=0)
    return np.stack([y_map, x_map], axis=0)


class DoubleConvBinary(nn.Module):
    def __init__(self, c_in: int, c_out: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class BinaryUNetSmall(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, base: int = 32):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.e1 = DoubleConvBinary(in_channels, base)
        self.e2 = DoubleConvBinary(base, base * 2)
        self.e3 = DoubleConvBinary(base * 2, base * 4)
        self.e4 = DoubleConvBinary(base * 4, base * 8)
        self.b = DoubleConvBinary(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, kernel_size=2, stride=2)
        self.d4 = DoubleConvBinary(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, kernel_size=2, stride=2)
        self.d3 = DoubleConvBinary(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.d2 = DoubleConvBinary(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.d1 = DoubleConvBinary(base * 2, base)
        self.head = nn.Conv2d(base, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], dim=1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        return self.head(d1)


class DoubleConv(nn.Module):
    def __init__(self, c_in: int, c_out: int, dropout: float = 0.0):
        super().__init__()
        layers = [
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class UNetEnhanced(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, base: int = 48, dropout: float = 0.10):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.e1 = DoubleConv(in_channels, base, dropout=0.0)
        self.e2 = DoubleConv(base, base * 2, dropout=0.0)
        self.e3 = DoubleConv(base * 2, base * 4, dropout=0.0)
        self.e4 = DoubleConv(base * 4, base * 8, dropout=dropout * 0.5)
        self.b = DoubleConv(base * 8, base * 16, dropout=dropout)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, kernel_size=2, stride=2)
        self.d4 = DoubleConv(base * 16, base * 8, dropout=dropout * 0.5)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, kernel_size=2, stride=2)
        self.d3 = DoubleConv(base * 8, base * 4, dropout=0.0)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.d2 = DoubleConv(base * 4, base * 2, dropout=0.0)
        self.u1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.d1 = DoubleConv(base * 2, base, dropout=0.0)
        self.head = nn.Conv2d(base, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], dim=1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        return self.head(d1)


class ConvBlock(nn.Module):
    def __init__(self, c_in: int, c_out: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class LastVisibleEstimator(nn.Module):
    def __init__(self, aux_dim: int, num_labels: int = 17, dropout: float = 0.25):
        super().__init__()
        self.image_encoder = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
        )
        self.image_pool = nn.AdaptiveAvgPool2d(1)
        self.image_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 192),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.aux_head = nn.Sequential(
            nn.Linear(aux_dim, 192),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(192, 96),
            nn.ReLU(inplace=True),
        )
        self.fusion = nn.Sequential(
            nn.Linear(192 + 96, 160),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(160, num_labels),
        )

    def forward(self, image: torch.Tensor, aux: torch.Tensor) -> torch.Tensor:
        image_feat = self.image_encoder(image)
        image_feat = self.image_pool(image_feat)
        image_feat = self.image_head(image_feat)
        aux_feat = self.aux_head(aux)
        fused = torch.cat([image_feat, aux_feat], dim=1)
        return self.fusion(fused)

## 4. Carga de modelos

In [ ]:
binary_model = BinaryUNetSmall(in_channels=1, out_channels=1).to(DEVICE)
binary_model.load_state_dict(torch.load(BINARY_MODEL_PATH, map_location=DEVICE))
binary_model.eval()

multiclass_model = UNetEnhanced(in_channels=3, out_channels=num_classes, base=48, dropout=0.10).to(DEVICE)
multiclass_model.load_state_dict(torch.load(MULTICLASS_MODEL_PATH, map_location=DEVICE))
multiclass_model.eval()

last_state_dict = torch.load(LAST_VISIBLE_MODEL_PATH, map_location=DEVICE)
inferred_aux_dim = int(last_state_dict['aux_head.0.weight'].shape[1])
inferred_num_labels = int(last_state_dict['fusion.3.weight'].shape[0])
last_model = LastVisibleEstimator(
    aux_dim=inferred_aux_dim,
    num_labels=inferred_num_labels,
    dropout=0.25,
).to(DEVICE)
last_model.load_state_dict(last_state_dict)
last_model.eval()

print('Models loaded. inferred_aux_dim=', inferred_aux_dim)

## 5. Features auxiliares y normalizacion del modelo final

Para el estimador final necesitamos reproducir la normalizacion de features
auxiliares usando las muestras de entrenamiento.

In [ ]:
def predict_binary_bbox_from_image(image_raw: np.ndarray) -> tuple[int, int, int, int] | None:
    image_resized = resize_image(image_raw, IMG_SIZE_BINARY).astype(np.float32) / 255.0
    image_tensor = torch.tensor(image_resized[None, None, ...], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        logits = binary_model(image_tensor)
        pred_mask_small = (torch.sigmoid(logits)[0, 0].detach().cpu().numpy() >= BINARY_THRESHOLD).astype(np.uint8)
    return bbox_from_mask(pred_mask_small, min_foreground_pixels=MIN_FOREGROUND_PIXELS)


def infer_multiclass_on_bbox(image_raw: np.ndarray, bbox: tuple[int, int, int, int]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    image_crop = crop_array(image_raw, bbox)
    image_crop = resize_image(image_crop, IMG_SIZE_MULTICLASS).astype(np.float32) / 255.0
    image_crop = normalize_image(image_crop)
    coords = build_coordinate_channels(IMG_SIZE_MULTICLASS[0], IMG_SIZE_MULTICLASS[1])
    image_channels = np.concatenate([np.expand_dims(image_crop, axis=0), coords], axis=0)
    image_tensor = torch.tensor(image_channels[None, ...], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        logits = multiclass_model(image_tensor)
        prob_map = torch.softmax(logits, dim=1)[0].detach().cpu().numpy().astype(np.float32)
        pred_mask = torch.argmax(logits, dim=1)[0].detach().cpu().numpy().astype(np.int64)
    return image_crop, pred_mask, prob_map


def extract_aux_features_from_pred_mask(pred_mask: np.ndarray, prob_map: np.ndarray) -> np.ndarray:
    h, w = pred_mask.shape
    fg_mask = (pred_mask > 0).astype(np.float32)
    total_fg = float(fg_mask.sum()) + 1e-6

    presence = []
    area_ratio = []
    centroid_y = []
    y_min_norm = []
    y_max_norm = []
    height_span_norm = []
    mean_confidence = []
    for label in canonical_labels:
        class_id = label_to_class_id[label]
        class_mask = pred_mask == class_id
        area = float(class_mask.sum())
        presence.append(1.0 if area >= PRESENCE_THRESHOLD_PIXELS else 0.0)
        area_ratio.append(area / total_fg)
        if area > 0:
            ys, _ = np.where(class_mask)
            centroid_y.append(float(np.mean(ys) / max(h - 1, 1)))
            y_min_norm.append(float(np.min(ys) / max(h - 1, 1)))
            y_max_norm.append(float(np.max(ys) / max(h - 1, 1)))
            height_span_norm.append(float((np.max(ys) - np.min(ys) + 1) / max(h, 1)))
            mean_confidence.append(float(prob_map[class_id][class_mask].mean()))
        else:
            centroid_y.append(0.0)
            y_min_norm.append(0.0)
            y_max_norm.append(0.0)
            height_span_norm.append(0.0)
            mean_confidence.append(0.0)

    pred_present_indices = [i for i, p in enumerate(presence) if p > 0.5]
    min_present_idx = float(min(pred_present_indices)) if pred_present_indices else 0.0
    max_present_idx = float(max(pred_present_indices)) if pred_present_indices else 0.0
    num_present = float(len(pred_present_indices))

    row_profile = fg_mask.sum(axis=1).astype(np.float32)
    if row_profile.max() > 0:
        row_profile = row_profile / row_profile.max()
    binned_profile = np.array_split(row_profile, PROFILE_BINS)
    profile_features = [float(chunk.mean()) for chunk in binned_profile]

    return np.array(
        presence
        + area_ratio
        + centroid_y
        + y_min_norm
        + y_max_norm
        + height_span_norm
        + mean_confidence
        + [
            min_present_idx / (len(canonical_labels) - 1),
            max_present_idx / (len(canonical_labels) - 1),
            num_present / len(canonical_labels),
        ]
        + profile_features,
        dtype=np.float32,
    )


train_aux_features = []
for _, row in train_norm_df.iterrows():
    image_raw = read_gray(row['radiograph_path_abs'])
    gt_binary = build_binary_mask(row['binary_mask_path_abs'], size=None)
    bbox = bbox_from_mask(gt_binary, min_foreground_pixels=MIN_FOREGROUND_PIXELS)
    if bbox is None:
        bbox = (0, 0, image_raw.shape[1], image_raw.shape[0])
    else:
        bbox = expand_bbox(bbox, image_shape=image_raw.shape, pad_x=ROI_PAD_X, pad_y=ROI_PAD_Y)
    _, pred_mask, prob_map = infer_multiclass_on_bbox(image_raw, bbox)
    train_aux_features.append(extract_aux_features_from_pred_mask(pred_mask, prob_map))

if not train_aux_features:
    raise RuntimeError('No se pudieron construir features auxiliares para normalizacion del estimador final.')

train_aux_features = np.stack(train_aux_features, axis=0)
aux_mean = train_aux_features.mean(axis=0).astype(np.float32)
aux_std = train_aux_features.std(axis=0).astype(np.float32)
aux_std = np.where(aux_std < 1e-6, 1.0, aux_std)

print('train_aux_features:', train_aux_features.shape)

## 6. Inferencia del pipeline completo sobre imagenes nuevas

In [ ]:
def clip_mask_to_last_idx(mask_2d: np.ndarray, first_idx: int, last_idx: int) -> np.ndarray:
    first_idx = int(first_idx)
    last_idx = int(max(last_idx, first_idx))
    allowed_labels = canonical_labels[first_idx:last_idx + 1]
    allowed_ids = {label_to_class_id[label] for label in allowed_labels}
    out = np.zeros_like(mask_2d, dtype=np.int64)
    for class_id in allowed_ids:
        out[mask_2d == class_id] = class_id
    return out


def present_labels_from_mask(mask_2d: np.ndarray) -> list[str]:
    ids = sorted(int(x) for x in np.unique(mask_2d) if int(x) > 0)
    return [class_names[i] for i in ids]


def collect_input_images(input_dir: Path) -> list[Path]:
    return sorted(
        path for path in input_dir.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )


image_paths = collect_input_images(INFERENCE_IMAGE_DIR)
if not image_paths:
    if USE_DEMO_DATASET_IMAGES_IF_INPUT_EMPTY:
        demo_df = table.loc[
            table['usable_for_thoracolumbar_partial'] & ~table['needs_annotation_review']
        ].copy().sort_values(['split', 'image']).head(DEMO_IMAGE_COUNT)
        image_paths = [Path(path) for path in demo_df['radiograph_path_abs'].tolist() if Path(path).exists()]
        print('No se encontraron imagenes nuevas en inference_inputs. Se usaran imagenes demo del dataset:', len(image_paths))
    else:
        raise FileNotFoundError(f'No se encontraron imagenes soportadas en {INFERENCE_IMAGE_DIR}')

print('Images found:', len(image_paths))

In [ ]:
def save_mask_png(mask_2d: np.ndarray, path: Path) -> None:
    Image.fromarray(mask_2d.astype(np.uint8)).save(path)


def save_preview_figure(image_gray: np.ndarray, raw_mask: np.ndarray, clipped_mask: np.ndarray, out_path: Path, title: str) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image_gray, cmap='gray')
    axes[0].set_title(f'ROI\n{title}')
    axes[1].imshow(raw_mask, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
    axes[1].set_title('Raw multiclase')
    axes[2].imshow(clipped_mask, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
    axes[2].set_title('Clipped final')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.close(fig)


rows = []
preview_counter = 0

for image_path in image_paths:
    image_raw = read_gray(str(image_path))
    bbox_small = predict_binary_bbox_from_image(image_raw)
    if bbox_small is None:
        bbox = (0, 0, image_raw.shape[1], image_raw.shape[0])
        roi_source = 'pred_binary_fallback_full_image'
    else:
        bbox = scale_bbox(bbox_small, src_shape=IMG_SIZE_BINARY, dst_shape=image_raw.shape)
        bbox = expand_bbox(bbox, image_shape=image_raw.shape, pad_x=ROI_PAD_X, pad_y=ROI_PAD_Y)
        roi_source = 'pred_binary'

    image_roi, raw_pred_mask, raw_prob_map = infer_multiclass_on_bbox(image_raw, bbox)
    aux_features = extract_aux_features_from_pred_mask(raw_pred_mask, raw_prob_map)
    if aux_features.shape[0] != inferred_aux_dim:
        raise RuntimeError(
            f'Las features auxiliares calculadas tienen dimension {aux_features.shape[0]} '
            f'pero el checkpoint espera {inferred_aux_dim}. Revisa la coherencia entre 07 y 08.'
        )
    aux_features = ((aux_features - aux_mean) / aux_std).astype(np.float32)

    image_small = resize_image((image_roi * 255.0).astype(np.uint8), IMG_SIZE_LAST).astype(np.float32) / 255.0
    image_small = normalize_image(image_small)
    coords = build_coordinate_channels(IMG_SIZE_LAST[0], IMG_SIZE_LAST[1])
    image_tensor = torch.tensor(np.concatenate([np.expand_dims(image_small, axis=0), coords], axis=0)[None, ...], dtype=torch.float32, device=DEVICE)
    aux_tensor = torch.tensor(aux_features[None, ...], dtype=torch.float32, device=DEVICE)

    with torch.no_grad():
        last_logits = last_model(image_tensor, aux_tensor)
        last_pred_idx = int(torch.argmax(last_logits, dim=1)[0].detach().cpu().item())

    last_pred_idx = max(last_pred_idx, ASSUMED_FIRST_VISIBLE_IDX)
    clipped_mask = clip_mask_to_last_idx(raw_pred_mask, ASSUMED_FIRST_VISIBLE_IDX, last_pred_idx)

    raw_labels = present_labels_from_mask(raw_pred_mask)
    final_labels = present_labels_from_mask(clipped_mask)

    stem = image_path.stem
    raw_mask_path = OUTPUT_DIR / f'{stem}_raw_mask.png'
    final_mask_path = OUTPUT_DIR / f'{stem}_final_mask.png'
    preview_path = OUTPUT_DIR / f'{stem}_preview.png'

    if SAVE_MASKS:
        save_mask_png(raw_pred_mask, raw_mask_path)
        save_mask_png(clipped_mask, final_mask_path)

    if SAVE_FIGURES and preview_counter < MAX_VIS_PREVIEW:
        save_preview_figure(image_roi, raw_pred_mask, clipped_mask, preview_path, image_path.name)
        preview_counter += 1

    rows.append({
        'image_name': image_path.name,
        'image_path': str(image_path),
        'roi_source': roi_source,
        'bbox_x0': int(bbox[0]),
        'bbox_y0': int(bbox[1]),
        'bbox_x1': int(bbox[2]),
        'bbox_y1': int(bbox[3]),
        'predicted_first_visible_label': canonical_labels[ASSUMED_FIRST_VISIBLE_IDX],
        'predicted_last_visible_label': canonical_labels[last_pred_idx],
        'raw_present_labels': ', '.join(raw_labels),
        'final_present_labels': ', '.join(final_labels),
        'raw_num_labels': len(raw_labels),
        'final_num_labels': len(final_labels),
        'raw_mask_path': str(raw_mask_path),
        'final_mask_path': str(final_mask_path),
        'preview_path': str(preview_path),
    })

inference_df = pd.DataFrame(rows)
summary_path = OUTPUT_DIR / 'final_inference_summary.csv'
inference_df.to_csv(summary_path, index=False)

display(inference_df)
print('Guardado:', summary_path)

## 7. Vista rapida de resultados

In [ ]:
display(
    inference_df[
        ['image_name', 'predicted_last_visible_label', 'raw_num_labels', 'final_num_labels', 'final_present_labels']
    ].sort_values(['predicted_last_visible_label', 'final_num_labels'])
)

## 8. Conclusiones de uso

Este notebook representa la mejor forma actual de usar el pipeline:

- ROI binaria para ubicar la columna
- segmentacion multiclase para vertebras
- estimacion especializada del extremo visible
- clipping anatomico para reducir sobreprediccion

Si luego quieres convertir esto en una aplicacion o script productivo, esta es la
base recomendada.